# NB05 · Semana 10 — Primer DAG en Airflow

**Ingeniería de Datos — LCD · UCA Rosario · 2026**  
Versión: **para el alumno (a completar)**

## Objetivo

Llevar el pipeline (extracción → bronze → dbt → gold) a un **DAG de Airflow**: definir tareas, dependencias, *scheduling* y reintentos. Este notebook no ejecuta Airflow (eso ocurre en el contenedor `airflow-scheduler`): sirve para **construir y validar** el archivo del DAG antes de desplegarlo.

> Base de la **Entrega 3 (E3)**.

## 1. Anatomía de un DAG

Un DAG (grafo acíclico dirigido) declara tareas y el orden entre ellas. Escribimos el DAG a la carpeta `dags/`, que el scheduler de Airflow lee automáticamente.

## 2. Construir el DAG del proyecto

Completá la definición de tareas y sus dependencias. El DAG de referencia completo está en `dags/pipeline_bcra.py`.

In [1]:
# === Configuración del entorno — NO tocar ===
import os, sys
_d = os.getcwd()
while True:
    if os.path.exists(os.path.join(_d, "entorno.py")):
        sys.path.insert(0, _d); break
    _p = os.path.dirname(_d)
    if _p == _d:
        raise FileNotFoundError("No encuentro entorno.py subiendo desde el directorio actual.")
    _d = _p
from entorno import BASE, DAGS_DIR, info
info()
print("DAGs en:", DAGS_DIR)

Entorno detectado: servidor de la cátedra  (grupo: grupo07)
  Lake / bucket   : grupo07   →   bronze en s3://grupo07/bronze/series
  PostgreSQL      : base 'warehouse', usuario 'grupo07', host 'postgres'
  Material en     : /home/jovyan/work
DAGs en: /home/jovyan/work/dags


In [2]:
# El DAG completo. Esta celda solo lo guarda en memoria; la sección 4 lo escribe a dags/.
dag_code = r'''"""
DAG pipeline_grupo07 — URLhaus -> bronze (Delta en MinIO) -> dbt (silver/gold en DuckDB) -> PostgreSQL.

Gold publicada: fct_familias_mensual (atómica: por mes, tipo de amenaza, familia de malware
y tipo de archivo, sacados de los tags de URLhaus) y fct_amenazas_mensual (su resumen por tipo).

Ingeniería de Datos — LCD · UCA Rosario · 2026 · grupo07

Cómo corre en el servidor de la cátedra (Airflow 2.x):
  * El worker de Airflow NO trae deltalake, duckdb ni dbt. La tarea `preparar_entorno`
    crea UNA vez un virtualenv en /tmp/venv_grupo07 con todo lo necesario, y las demás
    tareas corren dentro de ese venv con ExternalPythonOperator (no se reinstala nada
    en cada corrida).
  * Airflow solo ve esta carpeta (dags/ del JupyterHub = /opt/airflow/dags/grupo07/).
    Por eso las credenciales se leen de `dags/.env` y dbt se toma de `dags/dbt/`.
  * Las credenciales se leen DENTRO de cada tarea: no se pasan por op_kwargs ni env=,
    así no quedan visibles en la pestaña "Rendered Template" de la UI de Airflow.

Backfill: cada corrida procesa SOLO su intervalo [data_interval_start, data_interval_end).
  extraer_bronze baja el volcado CSV de URLhaus (últimos 30 días o base completa), filtra
  las URLs registradas en ese intervalo y reemplaza en bronze solo esas filas. Correr dos
  veces el mismo intervalo da el mismo resultado. dbt y publicar_gold recalculan todo
  desde bronze, así que también son idempotentes.

Tareas:
  preparar_entorno -> [validar_postgres, extraer_bronze] -> dbt_build -> publicar_gold
"""
from __future__ import annotations

import os

import pendulum
from airflow.models.dag import DAG
from airflow.operators.bash import BashOperator
from airflow.operators.python import ExternalPythonOperator

AQUI = os.path.dirname(os.path.realpath(__file__))
ENV_FILE = os.path.join(AQUI, ".env")
DBT_SRC = os.path.join(AQUI, "dbt")
DBT_RUN = "/tmp/dbt_run_grupo07"          # copia de trabajo de dbt (dags/ puede ser solo lectura)
VENV = "/tmp/venv_grupo07"
VENV_PY = f"{VENV}/bin/python"
# Si cambiás esta lista, subí VENV_VERSION para que el venv se reconstruya.
VENV_REQS = "dbt-duckdb deltalake pandas pyarrow requests psycopg2-binary"
VENV_VERSION = "v1"


# ----------------------------------------------------------------------------------
# Las funciones de abajo corren dentro del venv (ExternalPythonOperator): Airflow
# copia SOLO el código de la función, así que cada una importa y define lo que usa.
# ----------------------------------------------------------------------------------
def _validar_postgres(env_file):
    """Valida credenciales de PostgreSQL y que haya un esquema donde podamos crear tablas."""
    import os
    import psycopg2

    for linea in open(env_file, encoding="utf-8"):
        linea = linea.strip()
        if linea and not linea.startswith("#") and "=" in linea:
            k, v = linea.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

    con = psycopg2.connect(host=os.environ.get("PGHOST", "postgres"),
                           port=os.environ.get("PGPORT", "5432"),
                           dbname=os.environ.get("PGDATABASE", "warehouse"),
                           user=os.environ["PGUSER"], password=os.environ["PGPASSWORD"],
                           connect_timeout=10)
    with con, con.cursor() as cur:
        cur.execute("select current_user, current_database(), version()")
        usuario, base, version = cur.fetchone()
        print(f"Conectado a '{base}' como '{usuario}' ({version.split(',')[0]})")
        cur.execute("""
            select nspname from pg_namespace
            where has_schema_privilege(nspname, 'CREATE')
              and nspname not like 'pg\\_%' and nspname <> 'information_schema'
            order by (nspname = current_user) desc, (nspname = 'public') desc, nspname
        """)
        esquemas = [r[0] for r in cur.fetchall()]
    con.close()
    pedido = os.environ.get("PG_SCHEMA")
    if pedido and pedido not in esquemas:
        raise RuntimeError(f"PG_SCHEMA={pedido} pero el usuario no puede crear tablas ahí. "
                           f"Esquemas con permiso CREATE: {esquemas}")
    if not esquemas:
        raise RuntimeError("El usuario no tiene permiso CREATE en ningún esquema: pedile a la "
                           "cátedra un esquema propio (p. ej. 'grupo07').")
    print(f"Esquemas donde se pueden crear tablas: {esquemas}. Se usará: {pedido or esquemas[0]}")


def _extraer_bronze(env_file, inicio, fin):
    """Ingesta URLhaus -> bronze (Delta) del intervalo [inicio, fin) que le pasa Airflow.

    Idempotente y apta para backfill: el resultado depende solo del intervalo, no de la
    fecha en que corre. Reescribe en bronze únicamente las filas de ese intervalo.
    """
    import csv
    import io
    import os
    import time
    import zipfile
    import pandas as pd
    import pyarrow as pa
    import requests
    from deltalake import DeltaTable, write_deltalake
    from deltalake.exceptions import TableNotFoundError

    for linea in open(env_file, encoding="utf-8"):
        linea = linea.strip()
        if linea and not linea.startswith("#") and "=" in linea:
            k, v = linea.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

    bronze = f"s3://{os.environ.get('MINIO_BUCKET', 'grupo07')}/bronze/series"
    opciones = {
        "AWS_ACCESS_KEY_ID": os.environ["AWS_ACCESS_KEY_ID"],
        "AWS_SECRET_ACCESS_KEY": os.environ["AWS_SECRET_ACCESS_KEY"],
        "AWS_ENDPOINT_URL": os.environ.get("AWS_ENDPOINT_URL", "http://minio:9000"),
        "AWS_REGION": os.environ.get("AWS_REGION", "us-east-1"),
        "AWS_ALLOW_HTTP": "true",
        "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    }
    clave = os.environ.get("URLHAUS_AUTH_KEY")
    if not clave:
        raise RuntimeError("Falta URLHAUS_AUTH_KEY en dags/.env")
    cabecera = {"Auth-Key": clave}

    # 1) El intervalo lo decide Airflow (data_interval_start / data_interval_end), no el reloj
    desde = pd.Timestamp(inicio).tz_convert("UTC")
    hasta = pd.Timestamp(fin).tz_convert("UTC")
    print(f"Intervalo a procesar: [{desde}, {hasta})")

    # 2) Fuente. La API /urls/recent/ solo devuelve los últimos 3 días (máx. 1000 URLs), así que
    #    no sirve para un backfill. URLhaus publica dos volcados CSV con todas las columnas:
    #    los últimos 30 días y la base completa. Se usa el más chico que cubre el intervalo.
    #    El reloj solo elige la fuente; el filtro de filas es siempre por el intervalo.
    #    La Auth-Key va dentro de la URL (así lo exige abuse.ch): nunca se imprime.
    ahora = pd.Timestamp.now(tz="UTC")
    base_exports = "https://urlhaus-api.abuse.ch/v2/files/exports/{clave}"
    url_reciente = os.environ.get("URLHAUS_CSV_RECIENTE", base_exports + "/recent.csv").format(clave=clave)
    url_completo = os.environ.get("URLHAUS_CSV_COMPLETO", base_exports + "/full.csv.zip").format(clave=clave)
    if desde >= ahora - pd.Timedelta(days=29):
        url_csv, archivo = url_reciente, "/tmp/urlhaus_grupo07/csv_recent"
    else:
        url_csv, archivo = url_completo, "/tmp/urlhaus_grupo07/csv_completo"

    # El volcado se guarda en /tmp y se reutiliza durante 6 horas: un backfill de muchos días
    # lo descarga una sola vez.
    os.makedirs(os.path.dirname(archivo), exist_ok=True)
    if not os.path.exists(archivo) or time.time() - os.path.getmtime(archivo) > 6 * 3600:
        visible = url_csv.replace(clave, "<AUTH-KEY>")
        print(f"Descargando {visible}")
        # Los errores de requests incluyen la URL (con la clave): se re-lanzan sin ella
        try:
            with requests.get(url_csv, headers=cabecera, timeout=600, stream=True) as res:
                if res.status_code != 200:
                    raise RuntimeError(f"URLhaus respondió {res.status_code} al bajar {visible}")
                with open(archivo + ".parcial", "wb") as f:
                    for bloque in res.iter_content(1 << 20):
                        f.write(bloque)
        except requests.RequestException as e:
            raise RuntimeError(f"Falló la descarga de {visible}: {type(e).__name__}") from None
        os.replace(archivo + ".parcial", archivo)

    # 3) Leer el CSV en streaming (la base completa es grande) y quedarse con el intervalo.
    #    Formato de URLhaus: líneas de comentario con '#', fechas 'YYYY-MM-DD HH:MM:SS' en UTC.
    columnas = ['id', 'date_added', 'url', 'url_status', 'last_online',
                'threat', 'tags', 'urlhaus_link', 'reporter']
    texto_desde = desde.strftime("%Y-%m-%d %H:%M:%S")
    texto_hasta = hasta.strftime("%Y-%m-%d %H:%M:%S")
    if zipfile.is_zipfile(archivo):
        zf = zipfile.ZipFile(archivo)
        crudo = zf.open(zf.namelist()[0])
    else:
        crudo = open(archivo, "rb")
    filas = []
    with io.TextIOWrapper(crudo, encoding="utf-8", errors="replace", newline="") as lineas:
        sin_comentarios = (l for l in lineas if not l.startswith("#"))
        for r in csv.reader(sin_comentarios):
            if len(r) < len(columnas) or not r[0].strip().isdigit():
                continue
            if texto_desde <= r[1] < texto_hasta:
                filas.append(r[:len(columnas)])
    print(f"{len(filas)} URLs de URLhaus en el intervalo")
    if not filas:
        print("Nada para escribir en este intervalo (bronze queda como estaba).")
        return 0

    # 4) Mismo esquema y tipos que la ingesta original por API (tags = lista de textos)
    df = pd.DataFrame(filas, columns=columnas)
    df["id"] = df["id"].astype("int64")
    df["date_added"] = pd.to_datetime(df["date_added"], utc=True)
    df["tags"] = df["tags"].map(lambda t: [x.strip() for x in t.split(",") if x.strip()] or None)
    for col in ["url", "url_status", "last_online", "threat", "urlhaus_link", "reporter"]:
        df[col] = df[col].where(df[col].str.strip().ne("") & df[col].ne("None"), None)
    df = df.drop_duplicates(subset=["id"])
    esquema = pa.schema([
        ("id", pa.int64()), ("date_added", pa.timestamp("us", tz="UTC")),
        ("url", pa.string()), ("url_status", pa.string()), ("last_online", pa.string()),
        ("threat", pa.string()), ("tags", pa.list_(pa.string())),
        ("urlhaus_link", pa.string()), ("reporter", pa.string()),
    ])
    tabla = pa.Table.from_pandas(df, schema=esquema, preserve_index=False)

    # 5) Reemplazar SOLO las filas del intervalo: correr dos veces el mismo día da lo mismo
    try:
        DeltaTable(bronze, storage_options=opciones)
        existe = True
    except TableNotFoundError:
        existe = False
    if existe:
        predicado = (f"date_added >= '{desde.strftime('%Y-%m-%dT%H:%M:%SZ')}' "
                     f"AND date_added < '{hasta.strftime('%Y-%m-%dT%H:%M:%SZ')}'")
        # schema_mode="merge": columnas que en la ingesta vieja quedaron sin tipo (todo None)
        # pasan a texto en vez de hacer fallar la escritura
        write_deltalake(bronze, tabla, mode="overwrite", predicate=predicado,
                        schema_mode="merge", storage_options=opciones)
    else:
        write_deltalake(bronze, tabla, mode="append", storage_options=opciones)
    # stg_urlhaus lee los .parquet directamente: se borran los archivos que el reemplazo
    # dejó fuera de la tabla, para que dbt no lea versiones viejas.
    DeltaTable(bronze, storage_options=opciones).vacuum(
        retention_hours=0, enforce_retention_duration=False, dry_run=False)
    print(f"Bronze: {len(df)} filas del intervalo escritas (ids {df['id'].min()}..{df['id'].max()}).")
    return len(df)


def _dbt_build(env_file, dbt_src, dbt_run, venv):
    """Copia el proyecto dbt a /tmp y corre `dbt build` (modelos + tests)."""
    import os
    import shutil
    import subprocess

    for linea in open(env_file, encoding="utf-8"):
        linea = linea.strip()
        if linea and not linea.startswith("#") and "=" in linea:
            k, v = linea.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

    shutil.rmtree(dbt_run, ignore_errors=True)
    shutil.copytree(dbt_src, dbt_run, ignore=shutil.ignore_patterns(
        "target", "logs", ".ipynb_checkpoints", "*.duckdb", "*.duckdb.wal"))

    env = dict(os.environ)
    env["HOME"] = "/tmp/home_grupo07"      # DuckDB guarda sus extensiones en $HOME
    os.makedirs(env["HOME"], exist_ok=True)
    env["PATH"] = f"{venv}/bin:" + env.get("PATH", "")
    r = subprocess.run([f"{venv}/bin/dbt", "build", "--profiles-dir", "."],
                       cwd=dbt_run, env=env, capture_output=True, text=True)
    print(r.stdout)
    print(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"dbt build terminó con código {r.returncode}")


def _publicar_gold(env_file, dbt_run):
    """Publica las tablas gold de DuckDB en PostgreSQL (lo que lee Metabase)."""
    import os
    import duckdb
    import psycopg2
    from psycopg2.extras import execute_values

    for linea in open(env_file, encoding="utf-8"):
        linea = linea.strip()
        if linea and not linea.startswith("#") and "=" in linea:
            k, v = linea.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

    # Cada tabla gold: columnas a leer de DuckDB (con su tipo) y la clave primaria.
    # El orden de las columnas es el mismo en el select, en el create y en el insert.
    TABLAS = {
        "fct_amenazas_mensual": {
            "columnas": [("anio", "integer"), ("mes", "integer"), ("tipo_amenaza", "text"),
                         ("cantidad_amenazas", "bigint")],
            "clave": ["anio", "mes", "tipo_amenaza"],
        },
        "fct_familias_mensual": {
            "columnas": [("fecha", "date"), ("anio", "integer"), ("mes", "integer"),
                         ("tipo_amenaza", "text"), ("familia_malware", "text"),
                         ("tipo_archivo", "text"), ("cantidad_amenazas", "bigint")],
            "clave": ["anio", "mes", "tipo_amenaza", "familia_malware", "tipo_archivo"],
        },
    }

    # 1) Leer gold de DuckDB (lo dejó dbt_build)
    ruta = os.path.join(dbt_run, "warehouse.duckdb")
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No existe {ruta}: ¿corrió dbt_build en este mismo worker?")
    datos = {}
    con = duckdb.connect(ruta, read_only=True)
    try:
        for nombre, t in TABLAS.items():
            select = ", ".join(f"cast({c} as {'varchar' if tipo == 'text' else tipo})"
                               for c, tipo in t["columnas"])
            datos[nombre] = con.sql(f"select {select} from {nombre}").fetchall()
            if not datos[nombre]:
                raise RuntimeError(f"{nombre} está vacía en DuckDB: no publico una tabla vacía.")
    finally:
        con.close()

    # 2) Escribir en PostgreSQL en UNA transacción: si algo falla, quedan las versiones anteriores
    pg = psycopg2.connect(host=os.environ.get("PGHOST", "postgres"),
                          port=os.environ.get("PGPORT", "5432"),
                          dbname=os.environ.get("PGDATABASE", "warehouse"),
                          user=os.environ["PGUSER"], password=os.environ["PGPASSWORD"],
                          connect_timeout=10)
    resultado = {}
    try:
        with pg, pg.cursor() as cur:
            esquema = os.environ.get("PG_SCHEMA")
            if not esquema:
                cur.execute("""
                    select nspname from pg_namespace
                    where has_schema_privilege(nspname, 'CREATE')
                      and nspname not like 'pg\\_%' and nspname <> 'information_schema'
                    order by (nspname = current_user) desc, (nspname = 'public') desc, nspname
                    limit 1
                """)
                fila = cur.fetchone()
                if not fila:
                    raise RuntimeError("Sin permiso CREATE en ningún esquema de PostgreSQL.")
                esquema = fila[0]
            for nombre, t in TABLAS.items():
                tabla = f'"{esquema}".{nombre}'
                columnas = ",\n".join(f"{c} {tipo} not null" for c, tipo in t["columnas"])
                chequeo = ", check (mes between 1 and 12)" if any(c == "mes" for c, _ in t["columnas"]) else ""
                # Si la tabla ya existe con otras columnas (cambió el modelo), se recrea.
                # Sin CASCADE: si una vista depende de ella, falla y avisa en vez de borrarla.
                cur.execute("""
                    select column_name from information_schema.columns
                    where table_schema = %s and table_name = %s
                """, (esquema, nombre))
                actuales = {r[0] for r in cur.fetchall()}
                esperadas = {c for c, _ in t["columnas"]} | {"actualizado_en"}
                if actuales and actuales != esperadas:
                    print(f"{tabla}: cambiaron las columnas ({sorted(actuales ^ esperadas)}), la recreo.")
                    cur.execute(f"drop table {tabla}")
                cur.execute(f"""
                    create table if not exists {tabla} (
                        {columnas},
                        actualizado_en timestamptz not null default now(),
                        primary key ({", ".join(t["clave"])}){chequeo}
                    )
                """)
                cur.execute(f"truncate {tabla}")
                lista = ", ".join(c for c, _ in t["columnas"])
                execute_values(cur, f"insert into {tabla} ({lista}) values %s", datos[nombre])
                cur.execute(f"select count(*), sum(cantidad_amenazas) from {tabla}")
                resultado[tabla] = cur.fetchone()
    finally:
        pg.close()
    for (tabla, (n, total)), nombre in zip(resultado.items(), TABLAS):
        if n != len(datos[nombre]):
            raise RuntimeError(f"Se esperaban {len(datos[nombre])} filas en {tabla} y hay {n}.")
        print(f"Publicadas {n} filas ({total} amenazas) en PostgreSQL: {tabla}")
    totales = {total for n, total in resultado.values()}
    if len(totales) != 1:
        raise RuntimeError(f"Las tablas gold no suman lo mismo: {resultado}")
    return sum(n for n, _ in resultado.values())


# ---------------------------------------------------------------------------- DAG
default_args = {
    "owner": "grupo07",
    "retries": 2,
    "retry_delay": pendulum.duration(minutes=1),
}

with DAG(
    dag_id="pipeline_grupo07",
    description="URLhaus -> bronze (Delta) -> dbt silver/gold -> PostgreSQL",
    start_date=pendulum.datetime(2024, 1, 1, tz="America/Argentina/Cordoba"),
    schedule="@daily",
    catchup=False,
    max_active_runs=1,          # dos corridas a la vez pisarían /tmp/dbt_run_grupo07
    default_args=default_args,
    tags=["proyecto-integrador", "grupo07"],
) as dag:

    preparar_entorno = BashOperator(
        task_id="preparar_entorno",
        bash_command=f"""
        set -e
        if [ ! -f {VENV}/.listo_{VENV_VERSION} ]; then
            rm -rf {VENV}
            python3 -m venv {VENV}
            {VENV}/bin/pip install --quiet --upgrade pip
            {VENV}/bin/pip install --quiet {VENV_REQS}
            touch {VENV}/.listo_{VENV_VERSION}
        fi
        {VENV_PY} -c "import deltalake, duckdb, pandas, psycopg2, requests; print('venv OK')"
        test -f {ENV_FILE} || (echo "Falta {ENV_FILE}: corré la celda de despliegue de NB05" && exit 1)
        """,
    )

    comunes = dict(python=VENV_PY, expect_airflow=False)

    validar_postgres = ExternalPythonOperator(
        task_id="validar_postgres", python_callable=_validar_postgres,
        op_kwargs={"env_file": ENV_FILE}, retries=0, **comunes)

    # El intervalo de cada corrida viaja como plantilla: Airflow la resuelve con las fechas de
    # ESA corrida (también en un backfill), y la tarea no mira el reloj para filtrar.
    extraer_bronze = ExternalPythonOperator(
        task_id="extraer_bronze", python_callable=_extraer_bronze,
        op_kwargs={"env_file": ENV_FILE,
                   "inicio": "{{ data_interval_start.isoformat() }}",
                   "fin": "{{ data_interval_end.isoformat() }}"},
        **comunes)

    dbt_build = ExternalPythonOperator(
        task_id="dbt_build", python_callable=_dbt_build,
        op_kwargs={"env_file": ENV_FILE, "dbt_src": DBT_SRC, "dbt_run": DBT_RUN, "venv": VENV},
        **comunes)

    publicar_gold = ExternalPythonOperator(
        task_id="publicar_gold", python_callable=_publicar_gold,
        op_kwargs={"env_file": ENV_FILE, "dbt_run": DBT_RUN}, **comunes)

    preparar_entorno >> [validar_postgres, extraer_bronze] >> dbt_build >> publicar_gold
'''
print(f"Código del DAG generado en memoria ({len(dag_code.splitlines())} líneas).")


Código del DAG generado en memoria (419 líneas).


## 3. Validar la sintaxis del DAG

Antes de desplegarlo, comprobamos que el código Python del DAG compila. Airflow no cargará un DAG con errores de sintaxis.

In [3]:
import ast
try:
    ast.parse(dag_code)
    print("✅ ¡Sintaxis perfecta! El código del DAG compila sin errores.")
except SyntaxError as e:
    print(f"❌ Error de sintaxis en la línea {e.lineno}: {e.msg}")
    raise

✅ ¡Sintaxis perfecta! El código del DAG compila sin errores.


## 3b. Configuración de dbt

NB05 también escribe `dbt_project.yml` y `profiles.yml` en `dbt/`, así el proyecto dbt queda igual en el JupyterHub (NB04) y en Airflow.

In [4]:
# Configuración de dbt que usa tanto NB04 (JupyterHub) como Airflow.
dbt_project_yml = r'''# Proyecto dbt del grupo07 — Ingeniería de Datos, LCD UCA 2026.
# Transforma bronze (URLhaus en MinIO) en silver (stg_urlhaus) y gold (fct_amenazas_mensual) sobre DuckDB.
name: 'proyecto_id'
version: '1.0.0'
config-version: 2
profile: 'ingdatos'

model-paths:  ["models"]
target-path:  "target"
clean-targets: ["target", "dbt_packages"]

# La clave tiene que ser igual a `name` (proyecto_id). Ambos modelos se materializan como tabla:
# stg_urlhaus (silver) y fct_amenazas_mensual (gold, la que se publica en PostgreSQL).
models:
  proyecto_id:
    +materialized: table
'''

profiles_yml = r'''ingdatos:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: warehouse.duckdb
      extensions:
        - httpfs
      settings:
        # Credenciales y endpoint de MinIO desde el entorno (JupyterHub o dags/.env en Airflow)
        s3_endpoint: "{{ env_var('S3_ENDPOINT', 'minio:9000') }}"
        s3_access_key_id: "{{ env_var('AWS_ACCESS_KEY_ID') }}"
        s3_secret_access_key: "{{ env_var('AWS_SECRET_ACCESS_KEY') }}"
        s3_region: "{{ env_var('AWS_REGION', 'us-east-1') }}"
        s3_use_ssl: "false"
        s3_url_style: "path"
        # date_added viene en UTC: sin esto, cast(... as timestamp) usa la zona de la máquina
        # y una URL del 1/12 00:00 UTC cae en noviembre si dbt corre con hora argentina.
        TimeZone: "UTC"
'''

dbt_dir = os.path.join(BASE, "dbt")
with open(os.path.join(dbt_dir, "dbt_project.yml"), "w", encoding="utf-8") as f:
    f.write(dbt_project_yml)
with open(os.path.join(dbt_dir, "profiles.yml"), "w", encoding="utf-8") as f:
    f.write(profiles_yml)
print("dbt_project.yml y profiles.yml escritos en", dbt_dir)

dbt_project.yml y profiles.yml escritos en /home/jovyan/work/dbt


## 3c. Modelos dbt

NB05 escribe también los modelos en `dbt/models/` y el test de cuadre en `dbt/tests/`.

* `stg_urlhaus` (silver) suma `familia_malware` y `tipo_archivo`, sacados de los **tags** de URLhaus: el campo `threat` casi siempre es `malware_download`, así que solo no alcanza para distinguir amenazas.
* `fct_familias_mensual` (gold nueva): amenazas por mes, familia y tipo de archivo, con una columna `fecha` para Metabase.
* `assert_gold_cuadra`: test que falla si la silver y las dos gold no suman lo mismo.

In [5]:
# Modelos dbt (silver y gold) + test de cuadre. Los usa NB04 y, copiados a dags/dbt, Airflow.
import shutil
stg_urlhaus_sql = r'''-- Capa silver: tipado, limpieza y deduplicado de amenazas.
-- Reglas: descartar filas sin id o sin fecha; una sola observación por ID.
-- De los tags de URLhaus se sacan la familia de malware y el tipo de archivo
-- (el campo threat casi siempre es 'malware_download', así que no alcanza para distinguir amenazas).
{{ config(materialized='table') }}

{#- Tags que indican el tipo de archivo que descarga la URL -#}
{%- set tipos_archivo = ['elf', 'exe', 'dll', 'sh', 'bash', 'bat', 'cmd', 'ps1', 'powershell',
                         'js', 'vbs', 'hta', 'lnk', 'msi', 'iso', 'img', 'zip', 'rar', '7z', 'gz',
                         'tar', 'doc', 'docx', 'xls', 'xlsx', 'pdf', 'apk', 'jar', 'py', 'php',
                         'html', 'txt', 'ascii'] -%}
{#- Tags genéricos que no son una familia: arquitectura, origen o comportamiento -#}
{%- set genericos = ['32-bit', '64-bit', 'arm', 'arm4', 'arm5', 'arm6', 'arm7', 'aarch64', 'mips',
                     'mipsel', 'mpsl', 'm68k', 'ppc', 'powerpc', 'sparc', 'sh4', 'x86', 'x86-64',
                     'x86_64', 'i586', 'i686', 'amd64', 'arc', 'opendir', 'encrypted', 'geofenced',
                     'botnetdomain', 'dropper', 'downloader', 'loader', 'payload', 'malware',
                     'malicious', 'script', 'stealer', 'rat', 'trojan', 'hfs', 'censys',
                     'unknown', 'none', 'null', 'nan'] -%}
{#- Tags de dónde estaba alojado el archivo (servicio o plataforma), tampoco son una familia -#}
{%- set hosting = ['github', 'gitlab', 'bitbucket', 'discord', 'dropbox', 'google-drive',
                   'googledrive', 'gdrive', 'onedrive', 'mediafire', 'mega', 'pastebin',
                   'telegram', 'cdn', 'wordpress', 'aws', 'azure', 'cloudflare', 'digitalocean',
                   'ngrok', 'ipfs', 'ftp', 'http', 'https', 'sourceforge', 'firebase'] -%}

with base as (
    select
        cast(id as integer) as id_amenaza,
        cast(date_added as timestamp) as fecha_agregado,
        cast(url as varchar) as url_maliciosa,
        cast(url_status as varchar) as estado,
        cast(threat as varchar) as tipo_amenaza,
        cast(reporter as varchar) as reportero,
        -- en bronze los tags son una lista (['Mozi', 'elf']) o texto: los pasamos a texto
        cast(tags as varchar) as etiquetas
    -- DuckDB lee directamente los archivos Parquet de tu Delta Lake en MinIO
    from read_parquet('s3://{{ env_var("MINIO_BUCKET", "grupo07") }}/bronze/series/*.parquet')
    where id is not null
      and date_added is not null
),
dedup as (
    select *,
        row_number() over (
            partition by id_amenaza
            order by fecha_agregado desc
        ) as rn
    from base
),
unicos as (
    select * from dedup where rn = 1
),
-- Una fila por tag, con su posición (el orden en que URLhaus los informa)
lista as (
    select id_amenaza,
           string_split(regexp_replace(coalesce(etiquetas, ''), '[\[\]''"]', '', 'g'), ',') as tags
    from unicos
),
tags as (
    select id_amenaza,
           trim(unnest(tags)) as tag,
           generate_subscripts(tags, 1) as pos
    from lista
),
clasificados as (
    select id_amenaza, pos,
        case when lower(tag) in ('{{ tipos_archivo | join("', '") }}') then lower(tag) end as archivo,
        case when tag = ''
               or lower(tag) in ('{{ tipos_archivo | join("', '") }}')
               or lower(tag) in ('{{ genericos | join("', '") }}')
               or lower(tag) in ('{{ hosting | join("', '") }}')
               or regexp_matches(tag, '^[A-Z]{2,3}$')        -- códigos de país: CHN, USA, ...
               or lower(tag) like 'dropped-by%'
               or lower(tag) like 'ua-%'
             then null else tag end as familia
    from tags
),
por_id as (
    -- el primer tag que es familia y el primero que es tipo de archivo
    select id_amenaza,
           arg_min(familia, pos) filter (where familia is not null) as familia_malware,
           arg_min(archivo, pos) filter (where archivo is not null) as tipo_archivo
    from clasificados
    group by id_amenaza
),
-- Mismo nombre con distintas mayúsculas (mirai / Mirai) = una sola familia:
-- se usa la forma más frecuente de cada nombre.
grafias as (
    select familia_malware,
           row_number() over (partition by lower(familia_malware)
                              order by count(*) desc, familia_malware) as orden
    from por_id
    where familia_malware is not null
    group by familia_malware
)
select
    u.id_amenaza,
    u.fecha_agregado,
    u.url_maliciosa,
    u.estado,
    u.tipo_amenaza,
    coalesce(g.familia_malware, 'sin_familia') as familia_malware,
    coalesce(p.tipo_archivo, 'desconocido') as tipo_archivo,
    u.etiquetas,
    u.reportero
from unicos u
left join por_id p using (id_amenaza)
left join grafias g
       on lower(g.familia_malware) = lower(p.familia_malware)
      and g.orden = 1
'''

fct_amenazas_mensual_sql = r'''-- Capa gold agregada: resumen de fct_familias_mensual por mes y tipo de amenaza.
-- Se calcula desde la gold atómica (no desde la silver), así las dos no se pueden desincronizar.

{{ config(materialized='table') }}

select
    anio,
    mes,
    tipo_amenaza,
    sum(cantidad_amenazas) as cantidad_amenazas
from {{ ref('fct_familias_mensual') }}
group by 1, 2, 3
order by anio desc, mes desc, cantidad_amenazas desc
'''

fct_familias_mensual_sql = r'''-- Capa gold atómica: el grano más fino que publicamos para el negocio / BI.
-- Grano: cantidad de amenazas por mes, tipo de amenaza, familia de malware y tipo de archivo.
-- Cada URL cuenta una sola vez. Los demás resúmenes (fct_amenazas_mensual) salen de esta tabla.

{{ config(materialized='table') }}

select
    cast(date_trunc('month', fecha_agregado) as date) as fecha,
    extract(year from fecha_agregado) as anio,
    extract(month from fecha_agregado) as mes,
    tipo_amenaza,
    familia_malware,
    tipo_archivo,
    count(id_amenaza) as cantidad_amenazas
from {{ ref('stg_urlhaus') }}
group by 1, 2, 3, 4, 5, 6
order by fecha desc, cantidad_amenazas desc
'''

schema_yml = r'''version: 2

models:
  - name: stg_urlhaus
    description: "Capa Silver: Amenazas deduplicadas y tipadas de URLhaus."
    columns:
      - name: id_amenaza
        tests:
          - not_null
          - unique
      - name: fecha_agregado
        tests:
          - not_null
      - name: familia_malware
        description: "Primer tag de URLhaus que nombra una familia de malware; 'sin_familia' si no hay."
        tests:
          - not_null
      - name: tipo_archivo
        description: "Tipo de archivo según los tags (elf, exe, sh, ...); 'desconocido' si no hay."
        tests:
          - not_null

  - name: fct_amenazas_mensual
    description: "Capa Gold agregada: amenazas por mes y tipo, derivada de fct_familias_mensual. Consumido por BI."
    columns:
      - name: anio
        tests: [not_null]
      - name: mes
        tests:
          - not_null
          - accepted_values:
              arguments:
                values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
      - name: tipo_amenaza
        tests: [not_null]

  - name: fct_familias_mensual
    description: "Capa Gold atómica: amenazas por mes, tipo de amenaza, familia de malware y tipo de archivo. Consumido por BI."
    columns:
      - name: fecha
        tests: [not_null]
      - name: mes
        tests:
          - not_null
          - accepted_values:
              arguments:
                values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
      - name: tipo_amenaza
        tests: [not_null]
      - name: familia_malware
        tests: [not_null]
      - name: tipo_archivo
        tests: [not_null]
'''

assert_gold_cuadra_sql = r'''-- Las dos tablas gold y la silver tienen que sumar lo mismo: cada URL cuenta una sola vez.
-- El test falla si esta consulta devuelve alguna fila.
with totales as (
    select
        (select count(*) from {{ ref('stg_urlhaus') }}) as silver,
        (select sum(cantidad_amenazas) from {{ ref('fct_amenazas_mensual') }}) as gold_tipo,
        (select sum(cantidad_amenazas) from {{ ref('fct_familias_mensual') }}) as gold_familia
)
select * from totales
where silver <> gold_tipo or silver <> gold_familia
'''

# Tests propios de dominio (además de los genéricos de schema.yml)
assert_fechas_validas_sql = r'''-- Regla de dominio: URLhaus empezó a registrar URLs en 2018 y no puede haber URLs registradas
-- en el futuro. Una fecha fuera de ese rango indica un error de parseo o de zona horaria en la
-- ingesta (por ejemplo, una fecha local tomada como UTC o un campo mal leído del CSV).
-- El test falla si esta consulta devuelve alguna fila.
select id_amenaza, fecha_agregado
from {{ ref('stg_urlhaus') }}
where fecha_agregado < timestamp '2018-01-01'
   or fecha_agregado > cast(now() as timestamp) + interval 1 day
'''

assert_familia_no_es_tag_generico_sql = r'''-- Regla de dominio: una familia de malware nunca puede ser un tag genérico de URLhaus
-- (arquitectura, tipo de archivo, hosting o país). Si aparece uno, la lista de tags genéricos
-- de stg_urlhaus quedó desactualizada y el ranking de familias del tablero estaría mal.
-- El test falla si esta consulta devuelve alguna fila.
select familia_malware, count(*) as urls
from {{ ref('stg_urlhaus') }}
where lower(familia_malware) in ('elf', 'exe', 'zip', 'sh', '32-bit', '64-bit', 'arm', 'mips',
                                 'x86', 'github', 'discord', 'dropbox', 'opendir', 'ua-wget')
   or regexp_matches(familia_malware, '^[A-Z]{2,3}$')
group by familia_malware
'''

assert_zona_horaria_utc_sql = r'''-- URLhaus publica date_added en UTC. Si DuckDB corre con otra zona horaria (por ejemplo la
-- de una máquina en Argentina), cast(... as timestamp) corre las fechas 3 horas y las URLs de
-- las últimas horas de cada mes se cuentan en el mes anterior. profiles.yml fija TimeZone=UTC.
-- El test falla si esta consulta devuelve alguna fila.
select current_setting('TimeZone') as zona_horaria
where current_setting('TimeZone') <> 'UTC'
'''

archivos = {
    "models/stg_urlhaus.sql": stg_urlhaus_sql,
    "models/fct_amenazas_mensual.sql": fct_amenazas_mensual_sql,
    "models/fct_familias_mensual.sql": fct_familias_mensual_sql,
    "models/schema.yml": schema_yml,
    "tests/assert_gold_cuadra.sql": assert_gold_cuadra_sql,
    "tests/assert_fechas_validas.sql": assert_fechas_validas_sql,
    "tests/assert_familia_no_es_tag_generico.sql": assert_familia_no_es_tag_generico_sql,
    "tests/assert_zona_horaria_utc.sql": assert_zona_horaria_utc_sql,
}
dbt_dir = os.path.join(BASE, "dbt")
for ruta, contenido in archivos.items():
    destino = os.path.join(dbt_dir, ruta)
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    with open(destino, "w", encoding="utf-8") as f:
        f.write(contenido)
    print("✅ escrito", destino)

# Jupyter crea .ipynb_checkpoints al abrir un .sql/.yml y dbt los toma como modelos: los borramos
for sub in ("models", "tests"):
    shutil.rmtree(os.path.join(dbt_dir, sub, ".ipynb_checkpoints"), ignore_errors=True)


✅ escrito /home/jovyan/work/dbt/models/stg_urlhaus.sql
✅ escrito /home/jovyan/work/dbt/models/fct_amenazas_mensual.sql
✅ escrito /home/jovyan/work/dbt/models/fct_familias_mensual.sql
✅ escrito /home/jovyan/work/dbt/models/schema.yml
✅ escrito /home/jovyan/work/dbt/tests/assert_gold_cuadra.sql
✅ escrito /home/jovyan/work/dbt/tests/assert_fechas_validas.sql
✅ escrito /home/jovyan/work/dbt/tests/assert_familia_no_es_tag_generico.sql


In [6]:
# Documentación para el repositorio: governance (6 secciones) y diseño del backfill
governance_md = r'''# Governance del dataset — Amenazas URLhaus (grupo07)

Ingeniería de Datos — LCD · UCA Rosario · 2026 · grupo07

## 1. Identificación

| Campo | Valor |
| --- | --- |
| Nombre | Amenazas URLhaus mensuales |
| Tablas publicadas | `warehouse.grupo07.fct_familias_mensual` (gold atómica), `warehouse.grupo07.fct_amenazas_mensual` (gold agregada), `warehouse.grupo07.v_amenazas_features` (vista de features) |
| Responsable | grupo07 (Ingeniería de Datos, LCD UCA Rosario 2026) |
| Propósito | Seguir la evolución mensual de URLs que distribuyen malware, por familia de malware y tipo de archivo, para priorizar detección y prevención |
| Consumidores | Tablero "Amenazas URLhaus — grupo07" en Metabase |
| Grano | `fct_familias_mensual`: una fila por (mes, tipo de amenaza, familia, tipo de archivo). `fct_amenazas_mensual`: una fila por (mes, tipo de amenaza) |
| Frecuencia | Diaria (`@daily`), DAG `pipeline_grupo07` en Airflow. `actualizado_en` indica la última publicación |

## 2. Origen

- **Fuente:** URLhaus, de abuse.ch. Es una base pública de URLs que distribuyen malware, cargada por investigadores de seguridad.
- **Extracción:** volcados CSV de URLhaus (últimos 30 días o base completa), con Auth-Key propia. Cada corrida de Airflow toma solo las URLs registradas en su intervalo `[data_interval_start, data_interval_end)`, así que la ingesta es reproducible y admite backfill.
- **Linaje:**
  1. `extraer_bronze` → bronze en Delta Lake (`s3://grupo07/bronze/series`, MinIO), una fila por URL, tal como la publica URLhaus.
  2. dbt `stg_urlhaus` (silver) → tipado, deduplicado por id, y derivación de `familia_malware` y `tipo_archivo` a partir de los tags.
  3. dbt `fct_familias_mensual` (gold atómica) → conteo por mes, tipo de amenaza, familia y tipo de archivo.
  4. dbt `fct_amenazas_mensual` (gold agregada) → suma de la atómica por mes y tipo de amenaza.
  5. `publicar_gold` → PostgreSQL (`warehouse.grupo07`), con reemplazo total en una sola transacción.
  6. Vista `v_amenazas_features` (NB06) → Metabase.

## 3. Diccionario

**`fct_familias_mensual`** (gold atómica)

| Columna | Tipo | Descripción |
| --- | --- | --- |
| `fecha` | date | Primer día del mes en que URLhaus registró la URL |
| `anio` | integer | Año de registro |
| `mes` | integer | Mes de registro (1–12) |
| `tipo_amenaza` | text | Campo `threat` de URLhaus (en la práctica, `malware_download`) |
| `familia_malware` | text | Primer tag que nombra una familia de malware, con las mayúsculas unificadas. `sin_familia` si no hay |
| `tipo_archivo` | text | Primer tag de tipo de archivo (`elf`, `exe`, `zip`, `sh`…). `desconocido` si no hay |
| `cantidad_amenazas` | bigint | URLs distintas (por id) registradas en el mes con esa combinación |
| `actualizado_en` | timestamptz | Momento de la publicación en PostgreSQL |

Clave primaria: (`anio`, `mes`, `tipo_amenaza`, `familia_malware`, `tipo_archivo`).

**`fct_amenazas_mensual`** (gold agregada): tiene `anio`, `mes`, `tipo_amenaza`, `cantidad_amenazas` y `actualizado_en`, con el mismo significado que arriba. Clave primaria: (`anio`, `mes`, `tipo_amenaza`).

**`v_amenazas_features`** (vista): además de las columnas de la agregada, suma `fecha`, `total_mes`, `pct_del_mes`, `cantidad_mes_anterior`, `variacion_mensual_pct`, `acumulado_tipo` y `promedio_movil_3m`.

Cada columna también tiene su descripción como `COMMENT` en PostgreSQL, y Metabase la muestra.

## 4. Sensibilidad

- **Clasificación:** dato público de inteligencia de amenazas. No contiene datos personales de víctimas.
- **Minimización:** a PostgreSQL y a Metabase solo llegan datos agregados. Las URLs maliciosas, los tags crudos y el usuario de URLhaus que reportó cada URL (`reporter`) quedan solo en bronze y silver.
- **Riesgo:** las URLs de bronze y silver apuntan a malware activo. No se publican como enlaces clicables ni se abren desde el tablero.
- **Credenciales:** la Auth-Key de URLhaus y las contraseñas de MinIO y PostgreSQL viven en variables de entorno y en `dags/.env`, que no se versiona. NB05 pide la API key con `getpass`, así que no queda escrita en ningún notebook. Las tareas leen las credenciales adentro, así que no aparecen en "Rendered Template" de Airflow.
- **Acceso:** Metabase lee con el usuario del grupo, y solo el esquema `grupo07`.

## 5. Calidad

| Control | Dónde | Qué detecta |
| --- | --- | --- |
| `not_null` + `unique` en `id_amenaza` | dbt, silver | URLs duplicadas o sin id |
| `not_null` en fechas, familia, tipo de archivo y claves | dbt, silver y gold | Filas incompletas |
| `accepted_values` de `mes` (1–12) | dbt, gold | Meses mal calculados |
| `assert_fechas_validas` (test propio) | dbt, silver | Fechas anteriores a 2018 o futuras: errores de parseo o de zona horaria |
| `assert_familia_no_es_tag_generico` (test propio) | dbt, silver | Una arquitectura, extensión, hosting o país contado como familia |
| `assert_gold_cuadra` (test propio) | dbt, gold | Silver y las dos gold que no suman lo mismo |
| Clave primaria y `check (mes between 1 and 12)` | PostgreSQL | Duplicados o meses inválidos al publicar |
| No publicar una gold vacía y verificar el conteo | DAG `publicar_gold` | Publicaciones incompletas |
| Reconciliación tabla = vista | NB06, sección 4 | Vista desincronizada |

Si un test de dbt falla, `dbt_build` falla y `publicar_gold` no corre: PostgreSQL conserva la última versión buena.

**Limitaciones conocidas:**
- La familia es aproximada, porque los tags los cargan quienes reportan, sin un formato fijo.
- `fecha` es cuándo URLhaus registró la URL, no cuándo ocurrió el ataque.
- El mes en curso está incompleto.

## 6. Licencia

- **Datos de URLhaus:** abuse.ch los publica para uso libre, incluido el comercial, bajo CC0 (dominio público). El acceso a la API y a los volcados (`urlhaus-api.abuse.ch/v2/files/exports/`) requiere una Auth-Key gratuita. Hay que citar a URLhaus/abuse.ch como fuente. *Verificar los términos vigentes en https://urlhaus.abuse.ch/ al momento de la entrega.*
- **Datos derivados (silver, gold, features):** se publican bajo los mismos términos que la fuente, citando URLhaus/abuse.ch.
- **Código del pipeline:** desarrollado por grupo07 para la materia Ingeniería de Datos (LCD UCA Rosario 2026), con fines académicos.
'''

backfill_md = r'''# Backfill del DAG `pipeline_grupo07`

Cada corrida procesa **solo su intervalo** `[data_interval_start, data_interval_end)`, el que le asigna Airflow. Ninguna tarea usa la fecha de hoy para decidir qué filas procesar, así que un backfill de enero de 2025 trae las URLs de enero de 2025.

## Por qué no usamos `/v1/urls/recent/`

El endpoint `/v1/urls/recent/` de la API de URLhaus no acepta fechas: siempre devuelve lo registrado en los últimos 3 días, con un máximo de 1000 URLs. Con él, un backfill traería las amenazas de hoy. Por eso `extraer_bronze` usa los **volcados CSV** que publica URLhaus, con las mismas columnas que la API:

| Intervalo pedido | Fuente |
| --- | --- |
| Dentro de los últimos 30 días | `https://urlhaus-api.abuse.ch/v2/files/exports/<AUTH-KEY>/recent.csv` |
| Más viejo | `https://urlhaus-api.abuse.ch/v2/files/exports/<AUTH-KEY>/full.csv.zip` (base completa) |

La Auth-Key va dentro de la URL, como lo pide abuse.ch, así que el DAG la reemplaza por `<AUTH-KEY>` en los logs y en los mensajes de error. Las direcciones viejas (`urlhaus.abuse.ch/downloads/...`) responden 410 Gone.

El reloj solo se usa para elegir cuál de los dos volcados descargar, porque el chico alcanza para lo reciente. Las filas siempre se filtran por el intervalo de Airflow. El volcado se guarda en `/tmp` durante 6 horas, así que un backfill de muchos días lo descarga una sola vez.

## Idempotencia

1. `extraer_bronze` toma del CSV las URLs con `dateadded` en el intervalo y **reemplaza en bronze solo las filas de ese intervalo**: `write_deltalake(mode="overwrite", predicate="date_added >= inicio AND date_added < fin")`. Correr dos veces el mismo intervalo deja el mismo resultado, sin duplicados.
2. `dbt_build` recalcula silver y gold desde todo bronze.
3. `publicar_gold` reemplaza las tablas de PostgreSQL en una transacción.

`max_active_runs=1` hace que las corridas de un backfill vayan de a una.

## Cómo correrlo

```bash
airflow dags backfill pipeline_grupo07 -s 2025-01-01 -e 2025-01-31
```

En la UI se ve una corrida por día, con su propio `data_interval_start` y `data_interval_end`. El log de `extraer_bronze` muestra `Intervalo a procesar: [...]` y cuántas URLs había en ese intervalo.
'''

docs_dir = os.path.join(BASE, "docs")
os.makedirs(docs_dir, exist_ok=True)
for nombre, contenido in {"GOVERNANCE.md": governance_md, "BACKFILL.md": backfill_md}.items():
    with open(os.path.join(docs_dir, nombre), "w", encoding="utf-8") as f:
        f.write(contenido)
    print("✅ escrito", os.path.join(docs_dir, nombre))


✅ escrito /home/jovyan/work/docs/GOVERNANCE.md
✅ escrito /home/jovyan/work/docs/BACKFILL.md


## 4. Desplegar

Escribimos el DAG a `dags/` y dejamos ahí todo lo que Airflow necesita. Airflow solo ve esa carpeta (`/opt/airflow/dags/grupo07/`): no ve el resto de `/home/jovyan/work`. Después de unos 5 minutos, activá el DAG en Airflow y disparalo manualmente (*Trigger*).

In [7]:
# Escribir el DAG en dags/ (la carpeta que Airflow ve como /opt/airflow/dags/grupo07/)
os.makedirs(DAGS_DIR, exist_ok=True)
destino = os.path.join(DAGS_DIR, "pipeline_grupo07.py")
with open(destino, "w", encoding="utf-8") as f:
    f.write(dag_code)
print("✅ DAG escrito en:", destino)

✅ DAG escrito en: /home/jovyan/work/dags/pipeline_grupo07.py


In [8]:
# Dejar dentro de dags/ todo lo que Airflow necesita: dbt, credenciales y .airflowignore
import ast, shutil, getpass

# 1) Validar la sintaxis del DAG
dag = os.path.join(DAGS_DIR, "pipeline_grupo07.py")
ast.parse(open(dag, encoding="utf-8").read())
print("1) DAG OK:", dag)

# 2) Que Airflow no lea copias de Jupyter, dbt ni scripts viejos
shutil.rmtree(os.path.join(DAGS_DIR, ".ipynb_checkpoints"), ignore_errors=True)
with open(os.path.join(DAGS_DIR, ".airflowignore"), "w") as f:
    f.write(".ipynb_checkpoints\ndbt\nscripts\n__pycache__\n")
print("2) .airflowignore escrito")

# 3) Copiar el proyecto dbt (work/dbt es el original, el que usa NB04)
shutil.copytree(os.path.join(BASE, "dbt"), os.path.join(DAGS_DIR, "dbt"), dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("target", "logs", ".ipynb_checkpoints",
                                              "*.duckdb", "*.duckdb.wal"))
print("3) dbt copiado a", os.path.join(DAGS_DIR, "dbt"))

# 4) Credenciales para Airflow: variables del JupyterHub + API key de URLhaus.
#    La API key se reutiliza del .env anterior o se pide por teclado: así no queda escrita
#    en el notebook ni en el repositorio.
ruta_env = os.path.join(DAGS_DIR, ".env")
anterior = {}
if os.path.exists(ruta_env):
    for linea in open(ruta_env, encoding="utf-8"):
        if "=" in linea:
            k, v = linea.strip().split("=", 1)
            anterior[k] = v
clave = (os.environ.get("URLHAUS_AUTH_KEY") or anterior.get("URLHAUS_AUTH_KEY")
         or getpass.getpass("API key de URLhaus: "))

claves = ["MINIO_BUCKET", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_ENDPOINT_URL",
          "PGHOST", "PGPORT", "PGDATABASE", "PGUSER", "PGPASSWORD"]
lineas = [f"{k}={os.environ[k]}" for k in claves if os.environ.get(k)]
lineas.append(f"URLHAUS_AUTH_KEY={clave}")
esquema = os.environ.get("PG_SCHEMA") or anterior.get("PG_SCHEMA")   # opcional
if esquema:
    lineas.append(f"PG_SCHEMA={esquema}")
with open(ruta_env, "w", encoding="utf-8") as f:
    f.write("\n".join(lineas) + "\n")
print("4) .env escrito con:", [l.split("=")[0] for l in lineas])

print("Contenido de dags/:", sorted(os.listdir(DAGS_DIR)))
print("Listo. Airflow lo toma en ~5 minutos.")


1) DAG OK: /home/jovyan/work/dags/pipeline_grupo07.py
2) .airflowignore escrito
3) dbt copiado a /home/jovyan/work/dags/dbt
4) .env escrito con: ['MINIO_BUCKET', 'AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_ENDPOINT_URL', 'PGHOST', 'PGPORT', 'PGDATABASE', 'PGUSER', 'PGPASSWORD', 'URLHAUS_AUTH_KEY']
Contenido de dags/: ['.airflowignore', '.env', '__pycache__', 'dbt', 'pipeline_grupo07.py', 'scripts']
Listo. Airflow lo toma en ~5 minutos.


## 5. Verificar la capa gold en PostgreSQL

Corré esta celda después de que el DAG termine en verde: `publicar_gold` crea y carga `fct_amenazas_mensual` (por tipo de amenaza) y `fct_familias_mensual` (por familia de malware y tipo de archivo) en el warehouse, las que lee Metabase. Las dos tienen que sumar la misma cantidad de amenazas.

In [9]:
# Después de correr el DAG en Airflow: verificar que las dos tablas gold llegaron a PostgreSQL
from sqlalchemy import create_engine, text
from entorno import pg_url
with create_engine(pg_url()).connect() as c:
    print(c.execute(text("select table_schema, table_name from information_schema.tables "
                         "where table_name in ('fct_amenazas_mensual', 'fct_familias_mensual')")).fetchall())
    for tabla in ("fct_amenazas_mensual", "fct_familias_mensual"):
        print(tabla, c.execute(text(f"select count(*), sum(cantidad_amenazas) from {tabla}")).one())
    print("Top familias:")
    for fila in c.execute(text("select familia_malware, sum(cantidad_amenazas) as n from fct_familias_mensual "
                               "group by 1 order by n desc limit 10")):
        print("  ", fila)
    print("Tipos de archivo:")
    for fila in c.execute(text("select tipo_archivo, sum(cantidad_amenazas) as n from fct_familias_mensual "
                               "group by 1 order by n desc limit 10")):
        print("  ", fila)

[('grupo07', 'fct_amenazas_mensual'), ('grupo07', 'fct_familias_mensual')]
fct_amenazas_mensual (90, Decimal('80597'))
fct_familias_mensual (2618, Decimal('80597'))
Top familias:
   ('LuaJIT-loader', Decimal('18385'))
   ('Mozi', Decimal('14091'))
   ('mirai', Decimal('12863'))
   ('sin_familia', Decimal('11784'))
   ('SmartLoader', Decimal('2580'))
   ('CoinMiner', Decimal('1645'))
   ('zapret', Decimal('1532'))
   ('CheatSheet', Decimal('1397'))
   ('SalatStealer', Decimal('1340'))
   ('FakeGit', Decimal('1221'))
Tipos de archivo:
   ('desconocido', Decimal('36262'))
   ('elf', Decimal('19267'))
   ('zip', Decimal('18349'))
   ('exe', Decimal('2876'))
   ('pdf', Decimal('1101'))
   ('sh', Decimal('967'))
   ('ascii', Decimal('747'))
   ('jar', Decimal('220'))
   ('rar', Decimal('167'))
   ('doc', Decimal('153'))


## 4. Desplegar

Escribí el DAG a `dags/` y verificá en la UI de Airflow (http://localhost:8080) que aparezca. Activá el DAG y disparalo manualmente para la primera corrida.

```python
with open('/work/dags/pipeline_grupoX.py','w') as f:
    f.write(dag_code)
```

> **Backfill y reintentos** (contenido de la semana 11): con `catchup=True` Airflow ejecuta las corridas históricas pendientes; `retries` reintenta las tareas que fallan.

In [10]:
import os, glob
dag = open(os.path.join(DAGS_DIR, "pipeline_grupo07.py"), encoding="utf-8").read()
print("DAG nuevo en dags/:", "fct_familias_mensual" in dag)
print("Modelo nuevo en dags/dbt:", glob.glob(os.path.join(DAGS_DIR, "dbt", "models", "fct_familias*")))

DAG nuevo en dags/: True
Modelo nuevo en dags/dbt: ['/home/jovyan/work/dags/dbt/models/fct_familias_mensual.sql']


In [11]:
import os, requests
clave = None
for l in open(os.path.join(DAGS_DIR, ".env"), encoding="utf-8"):
    if l.startswith("URLHAUS_AUTH_KEY="):
        clave = l.strip().split("=", 1)[1]
h = {"Auth-Key": clave}

r = requests.get("https://urlhaus.abuse.ch/downloads/csv_recent/", headers=h, timeout=30)
print("Respuesta del 410:\n", r.text[:600], "\n")

candidatas = [
    "https://urlhaus-api.abuse.ch/files/exports/recent.csv",
    "https://urlhaus-api.abuse.ch/files/exports/full.csv.zip",
    "https://urlhaus-api.abuse.ch/v1/files/exports/recent.csv",
    "https://urlhaus-api.abuse.ch/v2/files/exports/recent.csv",
    f"https://urlhaus-api.abuse.ch/v2/files/exports/{clave}/recent.csv",
    f"https://urlhaus-api.abuse.ch/v2/files/exports/{clave}/full.csv.zip",
]
for u in candidatas:
    try:
        r = requests.get(u, headers=h, timeout=60, stream=True)
        inicio = next(r.iter_content(300), b"")
        print(r.status_code, u.replace(clave, "<CLAVE>"), "\n   ", inicio[:200], "\n")
        r.close()
    except Exception as e:
        print("ERROR", u.replace(clave, "<CLAVE>"), e)

Respuesta del 410:
 <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01//EN" "http://www.w3.org/TR/html4/strict.dtd">
<html><head>
<title>410 Gone</title>
</head><body>
<h1>Gone</h1>
<p>The requested resource is no longer available on this server and there is no forwarding address.
Please remove all references to this resource.</p>
<hr>
<address>Apache Server at urlhaus.abuse.ch Port 443</address>
</body></html>
 

401 https://urlhaus-api.abuse.ch/files/exports/recent.csv 
    b'{"status":401,"message":"Missing auth token"}' 

401 https://urlhaus-api.abuse.ch/files/exports/full.csv.zip 
    b'{"status":401,"message":"Missing auth token"}' 

401 https://urlhaus-api.abuse.ch/v1/files/exports/recent.csv 
    b'{"status":401,"message":"Missing auth token"}' 

404 https://urlhaus-api.abuse.ch/v2/files/exports/recent.csv 
    b'{"status":404,"message":"page not found"}' 

200 https://urlhaus-api.abuse.ch/v2/files/exports/<CLAVE>/recent.csv 
    b'####################################################